In [10]:
import pandas as pd
import json

In [19]:
train = pd.read_csv('content/FE_train.csv')
test = pd.read_csv('content/FE_test.csv')

# Separate features and targets
target_cols = ['event', 'time_to_hit_hours']
id_col = 'event_id'

In [26]:
print(train.shape)
print(test.shape)

(221, 47)
(95, 45)


In [22]:
with open("feature_selection/final_feature_set.json", "r") as f:
    feature_data = json.load(f)

selected_features = feature_data["final_features"]

print(selected_features[:5])

['dist_min_ci_0_5h', 'dt_first_last_0_5h', 'multi_perimeter_flag', 'nonzero_dt_flag', 'alignment_abs']


In [23]:
train_selected = train[selected_features]
test_selected = test[selected_features]

In [24]:
print(train_selected.shape)
print(test_selected.shape)
assert list(train_selected.columns) == list(test_selected.columns)

(221, 30)
(95, 30)


In [40]:
events = df['event'].astype(bool)
print(df.loc[events, 'dist_min_ci_0_5h'].var())
print(df.loc[~events, 'dist_min_ci_0_5h'].var())

1036701.4283265022
32846082188.956192


In [41]:
from lifelines import CoxPHFitter

df = train[selected_features + target_cols]

cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col="time_to_hit_hours", event_col="event")

<lifelines.CoxPHFitter: fitted with 221 total observations, 152 right-censored observations>

In [43]:
surv_funcs = cph.predict_survival_function(test[selected_features])

In [47]:
print(surv_funcs[12])

0.001220     0.993532
0.007176     0.987064
0.064361     0.980621
0.130320     0.974190
0.132979     0.967612
               ...   
66.790141    0.342499
66.859891    0.342499
66.920463    0.059137
66.966919    0.059137
66.994474    0.059137
Name: 12, Length: 221, dtype: float64


In [48]:
horizons = [12, 24, 48, 72]

preds = {}
for H in horizons:
    preds[f"p_hit_{H}h"] = 1 - surv_funcs[H].values

In [49]:
print(preds)

{'p_hit_12h': array([0.00646756, 0.01293627, 0.01937861, 0.02581001, 0.03238784,
       0.03893918, 0.03893918, 0.04551724, 0.05211419, 0.05881605,
       0.06576363, 0.07285641, 0.0803619 , 0.08787403, 0.09540867,
       0.09540867, 0.10297996, 0.11058229, 0.11818975, 0.12580869,
       0.13345626, 0.13345626, 0.14168292, 0.15005297, 0.15878017,
       0.16768514, 0.17653434, 0.18567544, 0.19504603, 0.19504603,
       0.20479074, 0.21446188, 0.22406157, 0.23405404, 0.24434059,
       0.25453759, 0.26523848, 0.27599749, 0.2866631 , 0.29741035,
       0.30806232, 0.31878039, 0.32974595, 0.34058991, 0.35132735,
       0.36209358, 0.3728244 , 0.3835408 , 0.39462715, 0.40559277,
       0.41643475, 0.41643475, 0.4273852 , 0.4273852 , 0.43852684,
       0.43852684, 0.43852684, 0.44973435, 0.46080851, 0.47181131,
       0.48267428, 0.48267428, 0.49346766, 0.5041271 , 0.51464131,
       0.51464131, 0.51464131, 0.51464131, 0.52510281, 0.52510281,
       0.52510281, 0.52510281, 0.52510281, 0.525

In [51]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

y = Surv.from_dataframe("event", "time_to_hit_hours", df)

rsf = RandomSurvivalForest(
    n_estimators=300,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1
)

rsf.fit(df[selected_features], y)

,n_estimators,300
,max_depth,None
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,None


In [53]:
surv_funcs = rsf.predict_survival_function(df[selected_features])

In [ ]:
import numpy as np

def get_prob_at_horizon(surv_func, H):
    return 1 - surv_func(H)

pred_24 = np.array([get_prob_at_horizon(fn, 24) for fn in surv_funcs])